In [14]:
import pandas as pd
import re 
import nltk

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Azeroual\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [29]:
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Azeroual\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Azeroual\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [32]:
#Chargement de dataset 

df = pd.read_csv(
    "SMSSpamCollection",
    sep="\t",
    header=None,
    names=["label", "message"]
)
df.to_csv("SMSSpam.csv", index=False)

## <span style="color:#A23B72">Partie 1 — Nettoyage des Données</span>

In [33]:
df = pd.read_csv("SMSSpam.csv")

In [34]:
def cal_moyene() :
    nombres = df["word_count"] = df["message"].str.split().str.len().tolist()
    nombre = 0
    for e in nombres :
        nombre += e
    return(nombre / df.shape[0])

In [35]:
#Analyse 

nbr_totles = df.shape[0]
print("Nombres totales des examples",nbr_totles)
print("Répartition des classes" ,df['label'].unique())
print("Longueur moyenne des messages",cal_moyene())



Nombres totales des examples 5572
Répartition des classes ['ham' 'spam']
Longueur moyenne des messages 15.597451543431443


In [36]:
# Identi er la présence éventuelle de 

df["has_punctuation"] = df["message"].str.contains(r"[^\w\s]", regex=True)
df["has_digits"] = df["message"].str.contains(r"\d", regex=True)
df["has_url"] = df["message"].str.contains(r"http[s]?://|www\.", regex=True)
df["has_special_chars"] = df["message"].str.contains(r"[@#$%^&*()_+=<>]", regex=True)
df["has_uppercase"] = df["message"].str.contains(r"[A-Z]", regex=True)
df["has_emoji"] = df["message"].str.contains(r"[\U0001F600-\U0001F64F]", regex=True)

df.to_csv("SMSSpam.csv", index=False)

Nettoyage minimal à tester

In [37]:
#Mise en minuscules
df['message'] = df['message'].str.lower()
#Suppression de la ponctuation
df['message'] = df['message'].str.replace(r"[^\w\s]","",regex=True)
#Suppression des chiffres
df['message'] = df['message'].str.replace(r"[\d]", "",regex=True)
#Suppression des caractères spéciaux
df['message'] = df['message'].str.replace(r"[@#$%^&*()_+=<>]","", regex=True)

df.to_csv("SMSSpamMinimal.csv", index=False)

# Questions d'analyse

### Le nettoyage améliore-t-il systématiquement les performances ?

Oui, je vois que la longueur moyenne diminue.

### Peut-on supprimer des informations utiles en nettoyant trop agressivement ?

Oui, on supprime les doublons, les lettres dupliquées dans les mots.

In [38]:

#Supprimer répétition excessive des lettres
def remove_duplicate_charactere(text) :
    return re.sub(r"(.)\1{2,}", r"\1\1", text)

#Supprimer les mots doublés
def remove_duplicate_words(text) :
    words = text.split()
    unique_words = list(dict.fromkeys(words))
    return " ".join(unique_words)
df['message'] = df['message'].apply(remove_duplicate_words)
df['message'] = df['message'].apply(remove_duplicate_charactere)

df.to_csv("SMSSpamMinimal.csv", index=False)


## <span style="color:#A23B72">Partie 2 — Prétraitement Linguistique</span>

In [39]:
def remove_stopwords(words , stop_words):
    sans_stpwords = []
    for word in words :
        if word not in stop_words and len(word) >= 2:
            sans_stpwords.append(word)
    return sans_stpwords

def apply_Stemming(words,stemmer):
    return [stemmer.stem(word) for word in words]

def apply_lemmatization(words,lemmatizer):
    return [lemmatizer.lemmatize(word) for word in words]

In [ ]:
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
#Tokenisation

df['tokens'] = df['message'].str.split()
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

df['tokens'] = df['tokens'].apply(lambda words: remove_stopwords(words, stop_words))
df['tokens'] = df['tokens'].apply(lambda words: apply_Stemming(words, stemmer))
df['tokens'] = df['tokens'].apply(lambda words: apply_lemmatization(words, lemmatizer))
df.to_csv("SMSSpamAvance.csv", index=False)

# Questions d'analyse

## Quelle différence observez-vous entre stemming et lemmatisation ?

Les mots obtenus après la lemmatisation sont plus corrects et logiques que ceux du stemming. 

- Le stemming coupe simplement les mots sans tenir compte de leur sens
- La lemmatisation cherche la forme normale du mot (lemme) en utilisant un dictionnaire et l'analyse grammaticale

## La suppression des stopwords est-elle toujours pertinente dans un problème de spam ?

Non, la suppression des stopwords n'est pas toujours utile pour la détection de spam. 

Après une vérification manuelle, on constate que certains de ces mots courants peuvent aider à identifier les messages spam.

## <span style="color:#A23B72">Partie 3 — Vectorisation</span>

## <span style="color:#E23B55">Bag of Words (CountVectorizer)</span>

In [ ]:
#transformer les tokens en string car Bag of Words ateind des strings 